# SAE additive-steering smoke test (Gemma3-4b-it, layer 9, feature 3289 — "poetry")

Interactive companion to `steer_tester.py`. Loads Gemma3-4b-it via the toolkit's `GemmaPytorchInference`, attaches the layer-9 Gemma-scope JumpReLU SAE, and runs baseline + three additive-steering generations at increasing strengths.

Gemma3's post-norm architecture pushes per-token residual norms into the thousands, so steering strengths are on the same scale (`strength=1600` is strong; above ~2000 the model breaks down).

In [ ]:
# Make `interpret.*` importable + use the repo root as CWD so resource
# paths like `resources/experiments/...` resolve regardless of where the
# kernel was launched. Idempotent.
import os, sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "interpret" / "__init__.py").is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        if Path.cwd() != _p:
            os.chdir(_p)
        break


In [ ]:
from interpret.inference.gemma_pytorch import GemmaPytorchInference
from interpret.sae import HookManager, SAEConfig, SteeringMode, SteeringOp

In [ ]:
PROMPT = "What is your favorite job?"
LAYER = 9
STEERED_FEATURE = 3289    # Neuronpedia label: "poetry" (density 0.011)
STRENGTHS = [0.0, 800.0, 1200.0, 1600.0]

In [ ]:
wrapper = GemmaPytorchInference("google/gemma-3-4b-it")

manager = HookManager()
manager.add_sae(SAEConfig(layer_index=LAYER, device="mps"))

In [ ]:
for strength in STRENGTHS:
    manager.clear_steering()
    if strength != 0.0:
        manager.add_steering(
            SteeringOp(
                layer_index=LAYER,
                mode=SteeringMode.ADDITIVE,
                feature_index=STEERED_FEATURE,
                strength=strength,
                normalise=False,
            )
        )

    with manager.session(wrapper.model.model.layers):
        output = wrapper.generate(PROMPT, output_len=120, temperature=1)

    label = "BASELINE" if strength == 0.0 else f"STRENGTH={strength:g}"
    print(f"\n===== {label} =====")
    print(output.strip())